# तन्त्र Tantra — train on a free Kaggle GPU

1. On your PC: `python main.py --mode pack --with-checkpoint` → upload the files in `kaggle_upload/` as a Kaggle dataset named **tantra-data** (kaggle.com → Datasets → New dataset).
2. Here: **Add data** → your *tantra-data* dataset · **Settings → Accelerator → GPU** · **Run all**.
3. When it finishes, download **tantra_out.zip** (Output panel) and put `latest.pt` (and `best.pt`) into your PC's `Model/` folder. Training on your PC — or the next Kaggle session — continues from there.

A session stops cleanly before Kaggle's 12-hour limit (checkpoint saved every evaluation and at the end).

In [ ]:
# ── settings ──────────────────────────────────────────────
STAGE  = "pretrain"   # "pretrain" first (learn language), later "sft" (learn to answer)
STEPS  = 30000        # target step (continues from the uploaded latest.pt if there is one)
BATCH  = 32           # sequences per step on the GPU (lower to 16 if you get 'out of memory')
HOURS  = 11           # stop cleanly before Kaggle's 12 h limit
BRANCH = "main"       # Tantra code branch to use

In [ ]:
import os, glob, shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "⚠ No GPU — Settings → Accelerator → GPU")
if not os.path.isdir("/kaggle/working/Tantra-LLM"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/atulyaai/Tantra-LLM.git",
                    "/kaggle/working/Tantra-LLM"], check=True)
os.chdir("/kaggle/working/Tantra-LLM")
# PyTorch is already on Kaggle; install the rest
subprocess.run("pip install -q tokenizers psutil fastapi uvicorn python-multipart", shell=True, check=True)
print("code ready:", os.getcwd())

In [ ]:
# ── connect the uploaded data ─────────────────────────────
found = glob.glob("/kaggle/input/**/pretrain.jsonl", recursive=True) + glob.glob("/kaggle/input/**/sft.jsonl", recursive=True)
assert found, "Add your 'tantra-data' dataset: Add data (right panel) → Your Datasets"
src = os.path.dirname(found[0])
os.makedirs("Datasets", exist_ok=True); os.makedirs("Model", exist_ok=True)
for name in os.listdir(src):
    s = os.path.join(src, name)
    if name.endswith(".jsonl"):
        dst = os.path.join("Datasets", name)
        if not os.path.exists(dst): os.symlink(s, dst)            # big data: link, no copy
    elif name in ("tokenizer.json", "latest.pt", "latest.pt.meta.json", "training_status.json", "probe_history.jsonl"):
        shutil.copy(s, os.path.join("Model", name))                 # copied: training writes these
print(sorted(os.listdir("Datasets")), sorted(os.listdir("Model")))
RESUME = os.path.exists("Model/latest.pt")
print("continuing from Model/latest.pt" if RESUME else "starting a new model")

In [ ]:
# ── train (progress below; checkpoints saved at every evaluation) ─
cmd = (f"python main.py --mode train --stage {STAGE} --device cuda --batch-size {BATCH} --grad-accum 1 "
       f"--seq-len 512 --steps {STEPS} --warmup 500 --eval-every 500 --log-every 50 --workers 2 --max-hours {HOURS}"
       + ("" if RESUME else " --fresh"))
print(cmd)
!{cmd}

In [ ]:
# ── pack the result for download ──────────────────────────
os.makedirs("/kaggle/working/tantra_out", exist_ok=True)
for n in ("latest.pt", "best.pt", "latest.pt.meta.json", "best.pt.meta.json", "training_status.json", "probe_history.jsonl"):
    if os.path.exists(f"Model/{n}"): shutil.copy(f"Model/{n}", "/kaggle/working/tantra_out/")
shutil.make_archive("/kaggle/working/tantra_out", "zip", "/kaggle/working/tantra_out")
shutil.rmtree("/kaggle/working/Tantra-LLM/Datasets", ignore_errors=True)   # keep the output small
print("Download /kaggle/working/tantra_out.zip → put latest.pt and best.pt into your PC's Model/ folder")
print(open("Model/training_status.json").read()[:400] if os.path.exists("Model/training_status.json") else "")